# 08 — Binary microlensing

Map of the **binary point-mass lens**: caustic structure depends on `(d, q_m)` — with the (close, intermediate, wide) topological regimes — and a single source crossing produces sharp light-curve spikes used for exoplanet detection in microlensing surveys.

References: Schneider+ Saas-Fee 33 (2006), Sec. 6.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


## 1. Magnification maps for several (d, q) regimes

In [ ]:
cases = [
    ('close', 0.6, 0.5),
    ('resonant', 1.0, 0.5),
    ('wide', 1.6, 0.5),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (label, d, q_m) in zip(axes, cases):
    lens = gl.lens.BinaryPointMass(d=d, q_m=q_m)
    ax_axis, mu = lens.magnification_map(npix=300, halfwidth=2.0)
    im = ax.imshow(np.log10(mu.numpy() + 1e-3), origin='lower',
                   cmap='inferno',
                   extent=(-2, 2, -2, 2))
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{label}: d={d}, q={q_m}')
plt.tight_layout(); plt.show()


## 2. Caustic curves overlaid

In [ ]:
lens = gl.lens.BinaryPointMass(d=1.0, q_m=0.5)
ax_axis, detA = lens.critical_curves(n=400)
fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contour(ax_axis.numpy(), ax_axis.numpy(),
                detA.numpy(), levels=[0.0], colors='cyan')
ax.set_aspect('equal'); ax.set(xlabel=r'$x_1$', ylabel=r'$x_2$',
                                title='binary critical curves (detA = 0)')
plt.show()


## 3. Light curve from a source trajectory

In [ ]:
# Source trajectory: straight line crossing the caustic.
t = torch.linspace(-1.0, 1.0, 200)
beta_x = t
beta_y = torch.full_like(t, 0.05)
# Inverse-ray-trace approach: for each beta, count how many image-plane
# pixels map to it (binary lens has up to 5 images, no closed form).
ax_axis, mu_map = lens.magnification_map(npix=600, halfwidth=2.0)
# Look up magnification along the trajectory.
ix = ((beta_x + 2.0) / 4.0 * 600).long().clamp(0, 599)
iy = ((beta_y + 2.0) / 4.0 * 600).long().clamp(0, 599)
mu_t = mu_map[iy, ix]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, mu_t)
ax.set(xlabel='source x position [theta_E]', ylabel=r'$\mu$',
       title='binary lens caustic-crossing light curve')
plt.show()
